# End-to-End Supervised Classification (Wine)

Walkthrough of the same pipeline as `src/train.py` / `src/predict.py`, with plots for class balance, confusion matrix, and model coefficients / feature importance.

**Dataset:** UCI Wine, bundled as `data/wine.csv` (13 chemical features → cultivar class 0/1/2).

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)
import joblib

RANDOM_SEED = 42
ROOT = Path("..").resolve()
DATA_PATH = ROOT / "data" / "wine.csv"
ARTIFACTS = ROOT / "artifacts"
TARGET = "target"
np.random.seed(RANDOM_SEED)
print("Ready. Data:", DATA_PATH)

## 1. Load data & light EDA

In [ ]:
df = pd.read_csv(DATA_PATH)
feature_cols = [c for c in df.columns if c != TARGET]
print(f"Shape: {df.shape}")
print(f"Missing: {df.isna().sum().sum()}")
print("\nClass balance:")
print(df[TARGET].value_counts().sort_index())
df[feature_cols].describe().T[["mean", "std", "min", "max"]].round(3)

In [ ]:
counts = df[TARGET].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.bar(counts.index.astype(str), counts.values, color=["#4C78A8", "#F58518", "#54A24B"])
ax.set_xlabel("Class (cultivar)")
ax.set_ylabel("Count")
ax.set_title("Class balance — Wine dataset")
for i, v in enumerate(counts.values):
    ax.text(i, v + 1, str(v), ha="center")
plt.tight_layout()
plt.show()

## 2. Train / val / test split (stratified, fixed seed)

In [ ]:
X = df[feature_cols].values
y = df[TARGET].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, random_state=RANDOM_SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_SEED, stratify=y_temp
)
print(f"train={len(y_train)}  val={len(y_val)}  test={len(y_test)}")

## 3. Preprocess — standardize (fit on train only)

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)
print("Scaler fitted on train only — no leakage into val/test.")

## 4. Train two models

In [ ]:
models = {
    "logistic_regression": LogisticRegression(max_iter=2000, random_state=RANDOM_SEED),
    "mlp": MLPClassifier(
        hidden_layer_sizes=(64, 32), activation="relu", solver="lbfgs",
        max_iter=2000, random_state=RANDOM_SEED,
    ),
}

def report(name, split, yt, yp):
    return {
        "model": name, "split": split,
        "accuracy": accuracy_score(yt, yp),
        "precision_macro": precision_score(yt, yp, average="macro", zero_division=0),
        "recall_macro": recall_score(yt, yp, average="macro", zero_division=0),
        "f1_macro": f1_score(yt, yp, average="macro", zero_division=0),
        "cm": confusion_matrix(yt, yp),
    }

rows, trained = [], {}
for name, model in models.items():
    model.fit(X_train_s, y_train)
    trained[name] = model
    for split, Xs, ys in [("val", X_val_s, y_val), ("test", X_test_s, y_test)]:
        rows.append(report(name, split, ys, model.predict(Xs)))

metrics_df = pd.DataFrame(rows).drop(columns=["cm"])
metrics_df


## 5. Confusion matrix (best model on test)

In [ ]:
best_name = max(
    [r for r in rows if r["split"] == "val"],
    key=lambda r: r["f1_macro"],
)["model"]
best = trained[best_name]
y_test_pred = best.predict(X_test_s)
cm = confusion_matrix(y_test, y_test_pred)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Test confusion matrix — {best_name}")
plt.tight_layout()
plt.show()
print(f"Best model (val macro-F1): {best_name}")
print(f"Test accuracy: {accuracy_score(y_test, y_test_pred):.4f}")
print(f"Test macro-F1: {f1_score(y_test, y_test_pred, average='macro'):.4f}")

## 6. Feature coefficients (Logistic Regression)

In [ ]:
lr = trained["logistic_regression"]
# Mean absolute coefficient across classes as a simple importance proxy
coef_importance = np.mean(np.abs(lr.coef_), axis=0)
order = np.argsort(coef_importance)[::-1]

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh([feature_cols[i] for i in order][::-1], coef_importance[order][::-1], color="#4C78A8")
ax.set_xlabel("Mean |coefficient| across classes")
ax.set_title("Logistic Regression — feature influence")
plt.tight_layout()
plt.show()

## 7. Persist artifacts (same as `train.py`)

In [ ]:
ARTIFACTS.mkdir(parents=True, exist_ok=True)
joblib.dump(best, ARTIFACTS / "best_model.joblib")
joblib.dump(scaler, ARTIFACTS / "scaler.joblib")
meta = {
    "best_model": best_name,
    "random_seed": RANDOM_SEED,
    "feature_columns": feature_cols,
    "target_column": TARGET,
}
(ARTIFACTS / "meta.json").write_text(json.dumps(meta, indent=2))
print("Wrote artifacts/", list(ARTIFACTS.glob("*")))

## What you practiced

- Offline tabular data + stratified splits
- Leakage-safe scaling (fit on train only)
- Comparing linear vs neural baselines with macro metrics
- Persisting a deployable `joblib` bundle for offline prediction

Next: try swapping in `RandomForestClassifier`, or add a simple calibration / threshold analysis.